# Pharmaceutical Document RAG

A high-performance pharmaceutical document question-answering system built on `RAGPipeline`.

---

| Component | Details |
|---|---|
| **Embedding** | sentence-transformers/all-MiniLM-L6-v2 (GPU-accelerated) |
| **Chunking** | Fixed-size (128 tokens, 16 overlap) |
| **Retrieval** | Hybrid  Vector + BM25, reciprocal rerank (top-3) |
| **LLM** | Mistral 7B Instruct (local, 4K context) |
| **OCR** | Tesseract (parallel, 200 DPI) |
| **UI** | Gradio |
| **Performance** | Index persistence + batched classification |

## 1. Install Dependencies

In [ ]:
import sys, subprocess, torch

# -- Auto-detect CUDA version from PyTorch -------------------------------
_cuda_ver = getattr(torch.version, "cuda", None)  # e.g. "12.4"
_cu_tag   = None

if _cuda_ver and torch.cuda.is_available():
    # Map "12.4" -> "cu124"
    _cu_tag = "cu" + _cuda_ver.replace(".", "")

# -- Install llama-cpp-python (pre-built CUDA wheel or CPU fallback) ----
# NOTE: --index-url (not --extra-index-url) forces pip to use the CUDA wheel
# index as the PRIMARY source, preventing fallback to the CPU build on PyPI.
# --extra-index-url https://pypi.org/simple lets pip find dependencies there.
_WHEEL_BASE = "https://abetlen.github.io/llama-cpp-python/whl"

if _cu_tag:
    _wheel_url = f"{_WHEEL_BASE}/{_cu_tag}"
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"CUDA : {_cuda_ver}  ->  using pre-built wheel [{_cu_tag}]")
    print(f"Index: {_wheel_url}\n")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "llama-cpp-python",
         "--index-url", _wheel_url,
         "--extra-index-url", "https://pypi.org/simple",
         "--force-reinstall", "--no-cache-dir"],
        check=True,
    )
else:
    print("No CUDA GPU detected - installing CPU-only llama-cpp-python")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "llama-cpp-python",
         "--force-reinstall", "--no-cache-dir"],
        check=True,
    )

# -- RESTART THE KERNEL after this cell, then re-run from cell 2. -------

# -- Other dependencies ---------------------------------------------------
%pip install -q pymupdf
%pip install -q llama-index llama-index-core
%pip install -q llama-index-embeddings-huggingface
%pip install -q llama-index-llms-llama-cpp
%pip install -q llama-index-llms-google-genai
%pip install -q llama-index-llms-ollama
%pip install -q llama-index-retrievers-bm25
%pip install -q sentence-transformers huggingface-hub
%pip install -q pytesseract pillow
%pip install -q "gradio>=6.9.0" --upgrade
%pip install -q "nest-asyncio>=1.6.0"

In [ ]:
# GPU Diagnostic - run this BEFORE and AFTER loading the model to confirm CUDA usage
import subprocess, sys

print("=== PyTorch CUDA ===")
import torch
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"torch.version.cuda:        {torch.version.cuda}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU:                       {torch.cuda.get_device_name(0)}")
    print(f"VRAM:                      {props.total_memory / 1e9:.1f} GB")
    print(f"Compute capability:        {props.major}.{props.minor}")
else:
    print("WARNING: No CUDA GPU - inference will run on CPU (slow)")

print("\n=== llama-cpp-python CUDA build check ===")
try:
    import llama_cpp
    print(f"version:                   {llama_cpp.__version__}")

    # llama_max_devices() > 1 -> CUDA build; == 1 -> CPU-only build
    max_devices = llama_cpp.llama_max_devices()
    gpu_ready = max_devices > 1
    status = "OK CUDA build" if gpu_ready else "X CPU-only build - reinstall with CUDA wheel (cell 1)"
    print(f"llama_max_devices():       {max_devices}  <- {status}")
    if not gpu_ready:
        print("\n  To fix: re-run cell 1 to install the CUDA wheel, then restart kernel.")
except Exception as e:
    print(f"import error: {e}")

print("\n=== VRAM usage (nvidia-smi) ===")
if torch.cuda.is_available():
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used,memory.free,memory.total",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True,
    )
    if r.returncode == 0:
        used, free, total = r.stdout.strip().split(", ")
        bar_len = 30
        used_frac = int(used) / int(total)
        bar = "#" * int(used_frac * bar_len) + "." * (bar_len - int(used_frac * bar_len))
        print(f"  [{bar}] {used}/{total} MiB used  ({free} MiB free)")
        print("  <- Re-run after loading the model to confirm VRAM increases (model on GPU)")
    else:
        print("  nvidia-smi not found - install NVIDIA drivers")

print("\n=== nvidia-smi summary ===")
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total,compute_cap",
     "--format=csv,noheader"],
    capture_output=True, text=True,
)
print(result.stdout.strip() if result.returncode == 0 else "nvidia-smi not found")

## 2. Imports

In [3]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path

# Ensure the src directory is on the path so rag can be imported
sys.path.insert(0, str(Path("..").resolve() / "src"))

from rag import RAGPipeline
import gradio as gr

c:\Users\anjoe\OneDrive\Desktop\Projects\PfizerExtern\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


resource module not available on Windows


## 3. Initialize RAG Pipeline

Loads the GGUF model directly from disk  no download needed.

> **GPU offload:** `n_gpu_layers=-1` offloads all layers to GPU (CUDA). Set `n_gpu_layers=0` to run on CPU only.

> **Index persistence:** `persist_dir="./storage"` saves the index to disk after building, enabling instant loading on subsequent runs.

In [ ]:
MODEL_PATH = r"C:\LLM Models\Mistral\mistral-7b-instruct-v0.2.Q4_K_M.gguf"

# n_gpu_layers=-1  -> offload ALL transformer layers to GPU (requires CUDA build).
# n_gpu_layers=0   -> CPU only (fallback if CUDA unavailable).
# The pipeline logs a warning at startup if llama-cpp-python lacks CUDA support.
rag = RAGPipeline(model_path=MODEL_PATH, persist_dir="./storage", n_gpu_layers=-1)
print("RAGPipeline initialized.")

# -- Post-load VRAM check -------------------------------------------------
# Re-run the GPU diagnostic cell after this to confirm VRAM increased,
# which proves the model layers are on the GPU and not in system RAM.
import subprocess, torch
if torch.cuda.is_available():
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True,
    )
    if r.returncode == 0:
        used, total = r.stdout.strip().split(", ")
        print(f"VRAM after model load: {used} / {total} MiB"
              f"  ({'model on GPU OK' if int(used) > 500 else 'low usage - check CUDA build'})")

## 4. Gradio UI

Upload a pharmaceutical PDF, click **Build Pipeline** to index it, then ask questions in the chat. Each answer includes source citations and per-chunk confidence scores.

**Performance optimizations:**
- GPU-accelerated embeddings for 5-10x faster indexing
- Fixed-size chunking (20-50x faster than semantic)
- Index persistence (instant loading after first build)
- Parallel OCR processing (3-5x faster for scanned pages)
- Reduced LLM context window for faster inference

In [ ]:
import time
import base64

_pipeline_ready = rag._query_engine is not None

_PHARMA_LABELS = {
    "cover_letter": "Cover Letter",
    "certificate_of_quality": "Certificate of Quality",
    "packaging_specification": "Packaging Specification",
    "bse_tse_declaration": "BSE / TSE Declaration",
    "material_description": "Material Description",
    "supplier_qualification": "Supplier Qualification",
    "chain_of_custody": "Chain of Custody",
    "unknown": "Unclassified",
    "unclassified": "Unclassified",
}


def _scan_label(ratio: float) -> str:
    if ratio <= 0:      return "Digital"
    if ratio < 0.5:     return f"Mixed ({ratio:.0%} scanned)"
    if ratio < 1.0:     return f"Mostly scanned ({ratio:.0%})"
    return "Scanned (OCR)"


def _format_index_display(stats: dict, doc_details: list) -> str:
    if not doc_details:
        return "*No index built.*"
    lines = [
        f"**{stats.get('total_files', 1)} file(s)** - "
        f"{stats['total_pages']}p - {stats['total_chunks']} chunks\n"
    ]
    if stats.get("classified") and stats.get("doc_type_counts"):
        lines.append("  \n".join(
            f"`{_PHARMA_LABELS.get(dt, dt)}` {c}p"
            for dt, c in sorted(stats["doc_type_counts"].items())
        ) + "\n")
    lines.append("---\n")
    for doc in doc_details:
        pharma_types = doc["pharma_doc_types"]
        type_str = (
            ", ".join(
                f"`{_PHARMA_LABELS.get(k, k)}`"
                for k, _ in sorted(pharma_types.items(), key=lambda x: -x[1])
            )
            if pharma_types else ""
        )
        lines.append(
            f"**{doc['file_name']}** - {doc['total_pages']}p - "
            f"{doc['total_chunks']}c - {_scan_label(doc['scan_ratio'])}"
            + (f"  \n{type_str}" if type_str else "")
            + "\n"
        )
    return "\n".join(lines)


def _build_source_card(source: dict) -> str:
    conf  = source["score"]
    cos   = source.get("cosine_similarity")
    label = _PHARMA_LABELS.get(source.get("pharma_doc_type", "unknown"), "Unclassified")
    scan  = _scan_label(1.0 if source["doc_type"] == "scanned" else 0.0)
    snippet = source["text"][:350]
    tail    = "..." if len(source["text"]) > 350 else ""
    conf_s  = f"{conf:.1f}%" if conf is not None else "--"
    cos_s   = f"{cos:.4f}" if cos is not None else "--"
    return (
        f"**`{source['file']}`** - p.{source['page']} - {label} - {scan}\n\n"
        f"conf {conf_s} - cos {cos_s}\n\n"
        f"> {snippet}{tail}"
    )


_PDF_EMPTY = (
    "<div style='display:flex;align-items:center;justify-content:center;"
    "height:100%;min-height:300px;color:#52525b;font-size:0.85rem;'>"
    "Upload a PDF to preview it here.</div>"
)


def get_pdf_viewer_html(pdf_path=None):
    if not pdf_path or not os.path.exists(pdf_path):
        return _PDF_EMPTY
    try:
        with open(pdf_path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode()
        return (
            '<iframe src="data:application/pdf;base64,' + b64 + '" width="100%" '
            'style="height:calc(100vh - 120px);min-height:400px;'
            'border:none;display:block;border-radius:6px;background:#fff;"></iframe>'
        )
    except Exception as exc:
        return f"<p style='color:#f87171;padding:12px;'>Error: {exc}</p>"


def on_file_upload(pdf_files):
    if not pdf_files:
        return _PDF_EMPTY, gr.update(choices=[], visible=False)
    paths = [f.name if hasattr(f, "name") else f for f in pdf_files if f is not None]
    if not paths:
        return _PDF_EMPTY, gr.update(choices=[], visible=False)
    choices = [(os.path.basename(p), p) for p in paths]
    return get_pdf_viewer_html(paths[0]), gr.update(
        choices=choices, value=paths[0], visible=len(choices) > 1
    )


def build_pipeline(pdf_files, accumulated_files):
    global _pipeline_ready

    if pdf_files is None:
        pdf_files = []
    if not isinstance(pdf_files, list):
        pdf_files = [pdf_files]

    new_paths = [
        f.name if hasattr(f, "name") else f
        for f in pdf_files if f is not None
    ]
    seen = {os.path.basename(p) for p in accumulated_files}
    for p in new_paths:
        if os.path.basename(p) not in seen:
            accumulated_files = accumulated_files + [p]
            seen.add(os.path.basename(p))

    _no = (
        "No files uploaded.",
        "0",
        "*Upload PDFs and click Build.*",
        accumulated_files,
        _PDF_EMPTY,
        gr.update(choices=[], visible=False),
    )
    if not accumulated_files:
        return _no

    try:
        _pipeline_ready = False
        if len(accumulated_files) == 1:
            rag.build(accumulated_files[0], classify_docs=True)
            label = os.path.basename(accumulated_files[0])
        else:
            rag.build_from_multiple_pdfs(
                accumulated_files, classify_docs=True,
                progress_callback=lambda c, t, f: None,
            )
            label = f"{len(accumulated_files)} files"

        _pipeline_ready = True
        stats = rag.get_stats()
        choices = [(os.path.basename(p), p) for p in accumulated_files]
        return (
            f"Ready - {label}",
            str(stats.get("total_files", len(accumulated_files))),
            _format_index_display(stats, rag.get_document_details()),
            accumulated_files,
            get_pdf_viewer_html(accumulated_files[0]),
            gr.update(choices=choices, value=accumulated_files[0], visible=len(choices) > 1),
        )
    except Exception as exc:
        import traceback
        _pipeline_ready = False
        return (
            f"Error: {exc}",
            "0",
            f"```\n{traceback.format_exc()}\n```",
            accumulated_files,
            _PDF_EMPTY,
            gr.update(choices=[], visible=False),
        )


def clear_files():
    global _pipeline_ready
    _pipeline_ready = False
    return [], "No index loaded.", "0", "*No index built yet.*", _PDF_EMPTY, gr.update(choices=[], visible=False)


_NO_SRC = ([], 0, "-- / --", "*No sources.*")


def ask(question, history, classify, expand, num_expansions, top_k, filter_doc_type):
    if not question.strip():
        return history, "", "*Ask a question above.*", *_NO_SRC, ""

    if not _pipeline_ready:
        return (
            history + [
                {"role": "user", "content": question},
                {"role": "assistant", "content": "Build an index first."},
            ],
            "", "*Pipeline not ready.*", *_NO_SRC, "",
        )

    orig_k = rag.similarity_top_k
    rag.similarity_top_k = top_k
    t0 = time.perf_counter()

    try:
        result = rag.query_with_sources(
            question, classify=classify, expand=expand, num_expansions=num_expansions,
        )
    except Exception as exc:
        rag.similarity_top_k = orig_k
        return (
            history + [
                {"role": "user", "content": question},
                {"role": "assistant", "content": f"Error: {exc}"},
            ],
            "", "*Retrieval error.*", *_NO_SRC, "",
        )

    gen_ms = round((time.perf_counter() - t0) * 1000, 1)
    rag.similarity_top_k = orig_k

    answer = result.get("answer", "")
    sources = result.get("sources", [])
    query_cat = result.get("query_category")

    new_history = history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer or "_Empty response - try again._"},
    ]

    if not answer.strip():
        return new_history, "", "*No answer generated.*", *_NO_SRC, ""

    if filter_doc_type and filter_doc_type != "all":
        sources = [s for s in sources if s.get("pharma_doc_type") == filter_doc_type]

    if not sources:
        lbl = _PHARMA_LABELS.get(filter_doc_type, filter_doc_type) if filter_doc_type != "all" else ""
        return new_history, "", f"*No sources{f' for `{lbl}`' if lbl else ''}.*", *_NO_SRC, f"{gen_ms} ms"

    parts = [f"**{len(sources)} chunk(s)**"]
    if query_cat:
        parts.append(f"query -> `{_PHARMA_LABELS.get(query_cat, query_cat)}`")
    if filter_doc_type and filter_doc_type != "all":
        parts.append(f"filter -> `{_PHARMA_LABELS.get(filter_doc_type, filter_doc_type)}`")

    cards = [_build_source_card(s) for s in sources]
    return (
        new_history, "",
        " - ".join(parts), cards, 0, f"1 / {len(cards)}", cards[0],
        f"{gen_ms} ms",
    )


def nav_source(cards, idx, direction):
    if not cards:
        return "*No sources.*", 0, "-- / --"
    new_idx = max(0, min(len(cards) - 1, idx + direction))
    return cards[new_idx], new_idx, f"{new_idx + 1} / {len(cards)}"


# --- CSS -----------------------------------------------------------------
CSS = """
html, body {
    height: 100vh !important;
    overflow: hidden !important;
}

.gradio-container {
    height: 100vh !important;
    overflow: hidden !important;
    padding: 4px 8px 0 !important;
    max-width: 100% !important;
    box-sizing: border-box !important;
}

/* Outer tab container: flex column filling 100vh */
#app-tabs {
    display: flex !important;
    flex-direction: column !important;
    height: calc(100vh - 4px) !important;
    overflow: hidden !important;
    min-height: 0 !important;
}

#app-tabs > div[role="tablist"],
#app-tabs > .tab-nav { flex-shrink: 0 !important; }

/* Only direct .tabitem children of #app-tabs get the flex-fill treatment.
   Using > prevents this rule from cascading into #side-tabs .tabitem. */
#app-tabs > .tabitem {
    flex: 1 !important;
    overflow: hidden !important;
    min-height: 0 !important;
}

/* Side panel: stack children from top, scroll when needed */
#side-panel {
    overflow-y: auto !important;
    overflow-x: hidden !important;
    min-height: 0 !important;
    justify-content: flex-start !important;
    align-self: flex-start !important;
    width: 100% !important;
}

/* Inner side tabs: natural height, no forced fill */
#side-tabs {
    display: flex !important;
    flex-direction: column !important;
    min-height: 0 !important;
    overflow: visible !important;
}

/* Space tabs evenly across the full width */
#side-tabs > div[role="tablist"],
#side-tabs .tab-nav {
    display: flex !important;
    width: 100% !important;
    justify-content: space-evenly !important;
}

#side-tabs > .tabitem {
    overflow-y: auto !important;
    overflow-x: hidden !important;
    min-height: 0 !important;
    padding: 4px 2px !important;
}

/* Document tab: no display override - Gradio sets display:none for hidden tabs */
#doc-tab {
    height: 100% !important;
    overflow: hidden !important;
}

#pdf-viewer { overflow: hidden !important; }

/* Source counter pill */
#src-counter textarea {
    text-align: center !important;
    font-size: 0.78rem !important;
    font-weight: 600 !important;
    background: transparent !important;
    border: none !important;
    color: #71717a !important;
    padding: 0 !important;
}
#src-counter {
    border: none !important;
    background: transparent !important;
    box-shadow: none !important;
}
"""

# --- Theme ---------------------------------------------------------------
THEME = gr.themes.Base(
    primary_hue=gr.themes.colors.indigo,
    neutral_hue=gr.themes.colors.zinc,
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"],
).set(
    body_background_fill="#09090b",
    body_text_color="#e4e4e7",
    body_text_color_subdued="#71717a",
    block_background_fill="#18181b",
    block_border_color="#27272a",
    block_border_width="1px",
    block_radius="8px",
    block_shadow="none",
    block_title_text_color="#e4e4e7",
    block_label_text_color="#a1a1aa",
    input_background_fill="#09090b",
    input_border_color="#3f3f46",
    input_border_color_focus="#818cf8",
    input_placeholder_color="#52525b",
    button_primary_background_fill="#6366f1",
    button_primary_background_fill_hover="#4f46e5",
    button_primary_text_color="#ffffff",
    button_secondary_background_fill="#27272a",
    button_secondary_border_color="#3f3f46",
    button_secondary_text_color="#e4e4e7",
    checkbox_label_text_color="#e4e4e7",
    checkbox_background_color_selected="#6366f1",
)

# --- Layout --------------------------------------------------------------
with gr.Blocks(title="Pharma Doc QA", css=CSS, theme=THEME) as demo:

    accumulated_files_state = gr.State([])
    sources_state = gr.State([])
    source_idx_state = gr.State(0)

    with gr.Tabs(elem_id="app-tabs"):

        # ==================================================================
        # Chat tab
        # ==================================================================
        with gr.Tab("Chat"):
            with gr.Row():

                # -- Chat column ---------------------------------------------
                with gr.Column(scale=5, min_width=360):
                    chatbot = gr.Chatbot(
                        height="calc(100vh - 155px)",
                        show_label=False,
                        elem_id="chatbot",
                        placeholder="Build an index, then ask a question.",
                    )
                    with gr.Row():
                        question_input = gr.Textbox(
                            placeholder="Ask a question about the document...",
                            show_label=False, scale=5, lines=1, max_lines=3, container=False,
                        )
                        ask_btn = gr.Button("Ask", variant="primary", scale=1, min_width=60)

                # -- Side panel ----------------------------------------------
                with gr.Column(scale=2, min_width=260, elem_id="side-panel"):

                    # Upload and controls (always visible)
                    pdf_input = gr.File(
                        label="Upload PDF(s)",
                        file_types=[".pdf"],
                        file_count="multiple",
                    )
                    with gr.Row():
                        build_btn = gr.Button("Build Index", variant="primary", size="sm", scale=3)
                        clear_btn = gr.Button("Clear", size="sm", scale=1)
                    with gr.Row():
                        status_box = gr.Textbox(
                            label="Status",
                            value="Ready - index loaded." if _pipeline_ready else "No index loaded.",
                            interactive=False, max_lines=1, scale=3,
                        )
                        doc_count_box = gr.Textbox(
                            label="Docs", value="0", interactive=False, max_lines=1, scale=1,
                        )
                        gen_time_box = gr.Textbox(
                            label="Gen", value="", interactive=False, max_lines=1, scale=1,
                        )

                    # One-at-a-time panels via nested tabs
                    with gr.Tabs(elem_id="side-tabs"):

                        with gr.Tab("Sources"):
                            source_header = gr.Markdown("*Sources appear here after querying.*")
                            with gr.Row(equal_height=True):
                                prev_btn = gr.Button("<", size="sm", scale=1, min_width=36)
                                source_counter = gr.Textbox(
                                    value="-- / --", show_label=False, interactive=False,
                                    container=False, scale=2, elem_id="src-counter",
                                )
                                next_btn = gr.Button(">", size="sm", scale=1, min_width=36)
                            source_card = gr.Markdown("*No sources yet.*")

                        with gr.Tab("Index"):
                            docs_display = gr.Markdown("*No index built yet.*")

                        with gr.Tab("Settings"):
                            top_k = gr.Slider(1, 20, value=5, step=1, label="Top-k chunks")
                            filter_doc_type = gr.Dropdown(
                                choices=[
                                    ("All types", "all"),
                                    ("Cover Letter", "cover_letter"),
                                    ("Certificate of Quality", "certificate_of_quality"),
                                    ("Packaging Specification", "packaging_specification"),
                                    ("BSE/TSE Declaration", "bse_tse_declaration"),
                                    ("Material Description", "material_description"),
                                    ("Supplier Qualification", "supplier_qualification"),
                                    ("Chain of Custody", "chain_of_custody"),
                                    ("Unclassified", "unknown"),
                                ],
                                value="all", label="Filter by doc type",
                            )
                            classify_query_toggle = gr.Checkbox(
                                label="Classify query", value=False,
                                info="Auto-detect relevant document category.",
                            )
                            expand_query_toggle = gr.Checkbox(
                                label="Expand query", value=False,
                                info="Generate paraphrases to improve recall.",
                            )
                            num_expansions = gr.Slider(1, 5, value=3, step=1, label="Expansions")

        # ==================================================================
        # Document Preview tab
        # ==================================================================
        with gr.Tab("Document", elem_id="doc-tab"):
            pdf_select = gr.Dropdown(
                label="File", choices=[], visible=False, interactive=True,
            )
            pdf_viewer = gr.HTML(value=_PDF_EMPTY, elem_id="pdf-viewer")

    # -- Event wiring ------------------------------------------------------
    _ask_in = [question_input, chatbot, classify_query_toggle, expand_query_toggle,
               num_expansions, top_k, filter_doc_type]
    _ask_out = [chatbot, question_input,
                source_header, sources_state, source_idx_state, source_counter, source_card,
                gen_time_box]

    pdf_input.change(on_file_upload, [pdf_input], [pdf_viewer, pdf_select])

    build_btn.click(
        build_pipeline, [pdf_input, accumulated_files_state],
        [status_box, doc_count_box, docs_display, accumulated_files_state, pdf_viewer, pdf_select],
    )
    clear_btn.click(
        clear_files, [],
        [accumulated_files_state, status_box, doc_count_box, docs_display, pdf_viewer, pdf_select],
    )
    pdf_select.change(lambda p: get_pdf_viewer_html(p), [pdf_select], [pdf_viewer])

    ask_btn.click(ask, _ask_in, _ask_out)
    question_input.submit(ask, _ask_in, _ask_out)

    prev_btn.click(lambda c, i: nav_source(c, i, -1),
                   [sources_state, source_idx_state], [source_card, source_idx_state, source_counter])
    next_btn.click(lambda c, i: nav_source(c, i, 1),
                   [sources_state, source_idx_state], [source_card, source_idx_state, source_counter])

demo.launch(share=False)